# APTOS 2019 Diabetic Retinopathy — Exploratory Data Analysis (EDA)

This notebook analyzes the APTOS 2019 Blindness Detection dataset and visualizes the impact of **Ben Graham Local Average Color Subtraction** and crop preprocessing.

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config import Config
from preprocess import ben_graham_preprocessing, crop_image_from_gray

Config.create_dirs()
df = pd.read_csv(Config.TRAIN_CSV)
print(f"Total training records: {len(df)}")
print(df.head())

## 1. Class Distribution Analysis

Diabetic retinopathy datasets typically exhibit significant class imbalance, with `Class 0 (No DR)` dominating.

In [ ]:
plt.figure(figsize=(8, 5))
counts = df['diagnosis'].value_counts().sort_index()
bars = plt.bar([f"Class {i}: {Config.CLASS_LABELS[i]}" for i in counts.index], counts.values, color=['#2b5c8f', '#4682b4', '#e67e22', '#d35400', '#c0392b'])
plt.title('APTOS 2019 DR Severity Class Distribution', fontsize=12, fontweight='bold')
plt.ylabel('Number of Images', fontsize=11)
plt.xticks(rotation=30, ha='right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, int(yval), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Visualizing Raw Fundus Images vs Ben Graham Preprocessed Images

Ben Graham preprocessing subtracts the local color average (Gaussian blur) to amplify microaneurysms, hemorrhages, and exudate visibility.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(10, 18))
fig.suptitle("Raw Fundus Images vs Ben Graham Preprocessed (Classes 0-4)", fontsize=14, fontweight='bold')

for cls in range(5):
    sample_id = df[df['diagnosis'] == cls].iloc[0]['id_code']
    img_path = os.path.join(Config.TRAIN_IMAGES_DIR, f"{sample_id}.png")
    if not os.path.exists(img_path):
        img_path = os.path.join(Config.TRAIN_IMAGES_DIR, f"{sample_id}.jpg")
        
    raw_bgr = cv2.imread(img_path)
    raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    processed = ben_graham_preprocessing(raw_rgb, img_size=Config.IMG_SIZE)
    
    axes[cls, 0].imshow(raw_rgb)
    axes[cls, 0].set_title(f"Class {cls} ({Config.CLASS_LABELS[cls]}): Raw", fontsize=10)
    axes[cls, 0].axis('off')
    
    axes[cls, 1].imshow(processed)
    axes[cls, 1].set_title(f"Class {cls}: Ben Graham Preprocessed", fontsize=10)
    axes[cls, 1].axis('off')

plt.tight_layout()
plt.show()